# NBA API — Initial EDA (Polars)

Exploring the free [`nba_api`](https://github.com/swar/nba_api) (no key required) to understand its structure and what data we can pull.

We'll use the `LeagueDashPlayerStats` endpoint — season-level totals for every player, the same source the statlas warehouse ingests from.

> `nba_api` returns **pandas** DataFrames, so we convert once with `pl.from_pandas(...)` and do all the EDA in **Polars**.

In [1]:
import nba_api
import polars as pl
from nba_api.stats.endpoints import leaguedashplayerstats

print("nba_api version:", nba_api.__version__)
print("polars version :", pl.__version__)

# Show all columns / wider tables when rendering
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)

nba_api version: 1.11.4
polars version : 1.41.2


polars.config.Config

## Pull a season of player stats

One call returns a pandas DataFrame of season totals for every player who logged minutes — we convert it straight to a Polars DataFrame.

In [2]:
SEASON = "2024-25"

resp = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON)
df = pl.from_pandas(resp.get_data_frames()[0])

print(f"Season {SEASON}: {df.height} players x {df.width} columns")

Season 2024-25: 569 players x 67 columns


## Basic info

Shape, column list, dtypes, and a peek at the data.

In [3]:
# All available columns
print(f"{df.width} columns:\n")
print(df.columns)

67 columns:

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT']


In [4]:
# Polars' equivalent of df.info() — column names, dtypes, and sample values
df.glimpse()

# Null counts per column (only show columns that actually have nulls)
nulls = df.null_count()
nonzero = [c for c in nulls.columns if nulls[c][0] > 0]
print("\nColumns with nulls:", {c: nulls[c][0] for c in nonzero} or "none")

Rows: 569
Columns: 67
$ PLAYER_ID             <i64> 1630639, 1631260, 1642358, 203932, 1628988, 1630174, 1630598, 1641745, 1641766, 1641737
$ PLAYER_NAME           <str> 'A.J. Lawson', 'AJ Green', 'AJ Johnson', 'Aaron Gordon', 'Aaron Holiday', 'Aaron Nesmith', 'Aaron Wiggins', 'Adam Flagler', 'Adama Sanogo', 'Adem Bona'
$ NICKNAME              <str> 'A.J.', 'AJ', 'AJ', 'Aaron', 'Aaron', 'Aaron', 'Aaron', 'Adam', 'Adama', 'Adem'
$ TEAM_ID               <i64> 1610612761, 1610612749, 1610612764, 1610612743, 1610612745, 1610612754, 1610612760, 1610612760, 1610612741, 1610612755
$ TEAM_ABBREVIATION     <str> 'TOR', 'MIL', 'WAS', 'DEN', 'HOU', 'IND', 'OKC', 'OKC', 'CHI', 'PHI'
$ AGE                   <f64> 24.0, 25.0, 20.0, 29.0, 28.0, 25.0, 26.0, 25.0, 23.0, 22.0
$ GP                    <i64> 26, 73, 29, 51, 62, 45, 76, 37, 4, 58
$ W                     <i64> 14, 44, 8, 33, 39, 29, 62, 34, 2, 12
$ L                     <i64> 12, 29, 21, 18, 23, 16, 14, 3, 2, 46
$ W_PCT                 <f64>

In [5]:
# A readable slice of the most common box-score columns
cols = ["PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "GP", "MIN",
        "PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT"]
df.select(cols).head(10)

PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,MIN,PTS,REB,AST,STL,BLK,FG_PCT,FG3_PCT
str,str,f64,i64,f64,i64,i64,i64,i64,i64,f64,f64
"""A.J. Lawson""","""TOR""",24.0,26,486.41,236,86,31,13,6,0.421,0.327
"""AJ Green""","""MIL""",25.0,73,1659.141667,541,174,108,37,7,0.429,0.427
"""AJ Johnson""","""WAS""",20.0,29,638.386667,220,59,76,12,3,0.385,0.267
"""Aaron Gordon""","""DEN""",29.0,51,1446.716667,748,247,164,23,14,0.531,0.436
"""Aaron Holiday""","""HOU""",28.0,62,792.246667,340,78,83,19,11,0.437,0.398
"""Aaron Nesmith""","""IND""",25.0,45,1121.618333,541,178,54,35,17,0.507,0.431
"""Aaron Wiggins""","""OKC""",26.0,76,1743.9,914,295,134,60,18,0.488,0.383
"""Adam Flagler""","""OKC""",25.0,37,202.965,65,27,12,8,3,0.26,0.194
"""Adama Sanogo""","""CHI""",23.0,4,21.498333,8,6,1,0,0,0.571,0.0


In [6]:
# Numeric summary
df.select(["AGE", "GP", "MIN", "PTS", "REB", "AST"]).describe()

statistic,AGE,GP,MIN,PTS,REB,AST
str,f64,f64,f64,f64,f64,f64
"""count""",569.0,569.0,569.0,569.0,569.0,569.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",26.230228,46.231986,1043.233743,492.108963,190.713533,114.783831
"""std""",4.220348,24.357135,805.522063,473.842651,176.520449,129.328428
"""min""",19.0,1.0,2.601667,0.0,0.0,0.0
"""25%""",23.0,25.0,314.276667,107.0,50.0,22.0
"""50%""",25.0,50.0,907.458333,356.0,158.0,75.0
"""75%""",29.0,68.0,1716.348333,732.0,273.0,156.0
"""max""",40.0,82.0,3036.076667,2484.0,1010.0,880.0


## Quick sanity check — top scorers

Totals, so sort by `PTS`. (For per-game or leaderboards we'd apply a minimum-games qualifier — see the warehouse reliability rule.)

In [7]:
top10 = df.sort("PTS", descending=True).head(10)
top10.select(["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "PTS"])

PLAYER_NAME,TEAM_ABBREVIATION,GP,PTS
str,str,i64,i64
"""Shai Gilgeous-Alexander""","""OKC""",76,2484
"""Anthony Edwards""","""MIN""",79,2177
"""Nikola Jokić""","""DEN""",70,2071
"""Giannis Antetokounmpo""","""MIL""",67,2036
"""Jayson Tatum""","""BOS""",72,1932
"""Devin Booker""","""PHX""",75,1923
"""Trae Young""","""ATL""",76,1841
"""Tyler Herro""","""MIA""",77,1840
"""Cade Cunningham""","""DET""",70,1830
